In [1]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.multivariate.manova import MANOVA
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import urllib.request
import io
from google.colab import drive

# Conectar ao Google Drive
drive.mount('/content/drive')

# Caminho onde o arquivo CSV será salvo no Google Drive
output_path = '/content/drive/My Drive/sumario-Neuroticismo-Abertura_A_Experiencia-Narcisismo-Psicopatia.csv'

# URL do dataset CSV (lembrando que o Google Sheets fornece uma URL específica para CSV)
dataset_url = 'https://docs.google.com/spreadsheets/d/1FUx1nRvhRRKhwYDwxUXMutXD-17lQ4I-pSZ27ERkC14/export?format=csv&gid=1161976353'

# Função para carregar o dataset a partir de uma URL
def load_data_from_url(url):
    try:
        response = urllib.request.urlopen(url)
        csv_data = response.read().decode('utf-8')
        dataset = pd.read_csv(io.StringIO(csv_data), delimiter=',')

        # Corrigir os separadores decimais de vírgula para ponto
        dataset.replace(',', '.', regex=True, inplace=True)

        # Converter colunas numéricas para o tipo float
        for col in dataset.columns:
            if col != 'Sexo':  # Ignorar a coluna "Sexo"
                dataset[col] = pd.to_numeric(dataset[col], errors='coerce')

        return dataset
    except urllib.error.HTTPError as e:
        print(f"Erro ao acessar a URL: {e}")
        return None
    except pd.errors.ParserError as e:
        print(f"Erro de parsing ao carregar o dataset: {e}")
        return None

# Função para calcular o sumário estatístico e gerar MANCOVA
def statistical_summary_and_mancova(df):
    if df is None or df.empty:
        print("DataFrame vazio ou inválido. Não é possível gerar o sumário.")
        return

    print('Sumário Estatístico Detalhado\n')

    # Filtrar por sexo
    males = df[df['Sexo'] == 'Masculino']
    females = df[df['Sexo'] == 'Femenino']

    # Variáveis de interesse
    variables = ['Neuroticismo', 'Abertura_A_Experiencia', 'Narcisismo', 'Psicopatia']

    # Cálculo de médias, desvios padrão e testes t
    for var in variables:
        print(f'\n{var}:\n')
        print('Média (Homens):', males[var].mean())
        print('Desvio Padrão (Homens):', males[var].std())
        print('Média (Mulheres):', females[var].mean())
        print('Desvio Padrão (Mulheres):', females[var].std())

        # Teste t entre homens e mulheres
        t_stat, p_value = stats.ttest_ind(males[var], females[var], nan_policy='omit')
        print(f'\nTeste t ({var}): t={t_stat:.4f}, p={p_value:.4f}')

    # Correlação entre Narcisismo e Psicopatia (para o conjunto de dados completo)
    correlation, p_value = stats.pearsonr(df['Narcisismo'].dropna(), df['Psicopatia'].dropna())
    print(f'\nCorrelação entre Narcisismo e Psicopatia: r={correlation:.4f}, p={p_value:.4f}')

    # MANCOVA - Análise Multivariada de Covariância
    formula = 'Neuroticismo + Abertura_A_Experiencia + Narcisismo + Psicopatia ~ Sexo'
    manova = MANOVA.from_formula(formula, data=df)
    print("\nMANCOVA Results:")
    print(manova.mv_test())

    # Post-Hoc Tukey HSD para cada variável
    for var in variables:
        print(f"\nPost-Hoc Results (Tukey HSD) for {var}:")
        tukey = pairwise_tukeyhsd(endog=df[var].dropna(), groups=df['Sexo'], alpha=0.05)
        print(tukey)

# Função para exportar os dados para CSV
def export_to_csv(df, filename=output_path):
    if df is None or df.empty:
        print("DataFrame vazio ou inválido. Não é possível exportar.")
        return

    df.to_csv(filename, index=False)
    print("Dados exportados para o arquivo:", filename)

# Função principal
def main():
    # Carregar o dataset a partir da URL
    df = load_data_from_url(dataset_url)

    if df is not None:
        # Sumário estatístico detalhado e MANCOVA
        statistical_summary_and_mancova(df)

        # Exportar os dados para CSV
        export_to_csv(df)
    else:
        print("Não foi possível carregar o dataset.")

# Executa a função principal
if __name__ == "__main__":
    main()

Mounted at /content/drive
Sumário Estatístico Detalhado


Neuroticismo:

Média (Homens): 53.334
Desvio Padrão (Homens): 14.659528031463468
Média (Mulheres): 52.5005
Desvio Padrão (Mulheres): 13.546228247628273

Teste t (Neuroticismo): t=0.1867, p=0.8529

Abertura_A_Experiencia:

Média (Homens): 63.334
Desvio Padrão (Homens): 17.604541698693907
Média (Mulheres): 63.3335
Desvio Padrão (Mulheres): 13.078977617777154

Teste t (Abertura_A_Experiencia): t=0.0001, p=0.9999

Narcisismo:

Média (Homens): 46.5275
Desvio Padrão (Homens): 7.031472762553113
Média (Mulheres): 46.94350000000001
Desvio Padrão (Mulheres): 8.49789402703616

Teste t (Narcisismo): t=-0.1687, p=0.8669

Psicopatia:

Média (Homens): 27.4995
Desvio Padrão (Homens): 7.58893826219244
Média (Mulheres): 20.555500000000002
Desvio Padrão (Mulheres): 10.903877859203167

Teste t (Psicopatia): t=2.3376, p=0.0248

Correlação entre Narcisismo e Psicopatia: r=0.0818, p=0.6157

MANCOVA Results:
                  Multivariate linear model
